# ボートレースAI予想 — スマホ（Google Colab）で実行

**PCがなくてもスマホのブラウザだけで動きます。** 各セルの左の▶を上から順にタップするだけ。

- Colab は `boatrace.jp` / `mbrace.or.jp` に通信できるので、公式データの取得・学習ができます。
- 予想は確率モデルに基づく**参考情報**です。的中は保証しません。舟券は20歳以上・自己責任で。

## 1. セットアップ（リポジトリ取得 + 解凍ライブラリ）

In [ ]:
!git clone -b claude/ocean-cup-31-ai-prediction-if4zwd https://github.com/kuruppe1/shogi-arena.git
%cd shogi-arena
!pip -q install lhafile
print('セットアップ完了')

## 2. すぐ予想を試す（学習なし・事前モデル）

まずは同梱サンプルで動作確認。

In [ ]:
!python -m boatrace_predictor.cli predict --demo

## 3. 実際のレースを予想（その日の公式番組表から）

`--fetch-date`（開催日）、`--venue`（会場名 例:児島 / 会場コード 例:16）、`--race`（レース番号）を指定。
まだ結果が出ていない当日・未来のレースでも、番組表が配布されていれば予想できます。

In [ ]:
# 例: 2026-07-27 の 児島 12R を予想（日付・会場・レースは書き換えてください）
!python -m boatrace_predictor.cli predict --fetch-date 2026-07-27 --venue 児島 --race 12

## 4.（任意）過去データで学習して精度を上げる

今年度〜昨日までの結果・水面・気象を取り込み、着順を教師データに係数を学習します。
期間が長いほど時間がかかります（まずは数週間で試すのがおすすめ）。

In [ ]:
# 4-1. 公式データを期間取得（休催日は自動スキップ）
!python -m boatrace_predictor.cli fetch --start 2026-07-01 --end 2026-07-26 --out history.json
# 4-2. 学習
!python -m boatrace_predictor.cli train --history history.json --out model.json

In [ ]:
# 4-3. 学習済みモデルで予想（水面・気象も加味）
!python -m boatrace_predictor.cli predict --fetch-date 2026-07-27 --venue 児島 --race 12 --model model.json

## （参考）実データなしで学習ロジックを検証

In [ ]:
!python -m boatrace_predictor.cli selftest